# US 5.3 -- Combined Selection Strategy

In the previous notebooks we explored two independent selection criteria:

- **Uncertainty** (US 5.1): pick samples the model is confused about.
- **Diversity** (US 5.2): pick samples that cover the target distribution.

Each criterion alone has blind spots:

| Strategy | Strength | Weakness |
|----------|----------|----------|
| Uncertainty only | Targets hard samples | May pick redundant similar samples |
| Diversity only | Good coverage | May pick easy samples the model already handles |

The **combined** strategy merges both signals into a single score:

$$\text{score}(x) = \alpha \cdot \text{uncertainty}(x) + (1 - \alpha) \cdot \text{diversity}(x)$$

where $\alpha \in [0, 1]$ controls the trade-off.

## What you will learn

1. How the combined score is computed
2. Effect of the alpha parameter
3. How to export the selection as a CSV
4. CLI usage for combined selection

## CLI equivalent

```bash
udm-epic5 select --strategy combined --alpha 0.5 --budget 50
```

## Prerequisites

- Python 3.12, PyTorch, scikit-learn
- Install: `uv pip install -e ".[epic5]"`

---
## 1. How the Combined Score Works

Both the uncertainty and diversity scores are first **normalised** to [0, 1]
using min-max scaling across the candidate pool.  Then they are combined:

$$\text{score}_i = \alpha \cdot \hat{u}_i + (1 - \alpha) \cdot \hat{d}_i$$

where:
- $\hat{u}_i$ is the normalised uncertainty score for sample $i$
- $\hat{d}_i$ is the normalised diversity score for sample $i$
- $\alpha$ is the mixing weight

We select the top-K samples by descending combined score.

### Special cases

- $\alpha = 1.0$: pure uncertainty selection (same as US 5.1)
- $\alpha = 0.0$: pure diversity selection (same as US 5.2)
- $\alpha = 0.5$: equal weight (recommended starting point)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from udm_epic4.models.dann import DANNModel
from udm_epic5.uncertainty.mc_dropout import enable_mc_dropout, mc_dropout_uncertainty, compute_entropy
from udm_epic5.selection.diversity import coreset_selection
from udm_epic5.selection.combined import combined_selection, export_selection_csv

In [ ]:
# Setup: create model and simulate a pool of target-domain images
model = DANNModel(
    backbone="convnext_tiny",
    pretrained=False,
    in_chans=3,
    decoder_channels=[256, 128, 64, 32],
    domain_head_hidden=256,
)

n_pool = 100
dummy_pool = torch.randn(n_pool, 3, 512, 512)
image_paths = [f"data/malaysia/images/img_{i:04d}.png" for i in range(n_pool)]

print(f"Pool size: {n_pool} target-domain images")

In [ ]:
# Step 1: Compute uncertainty scores
enable_mc_dropout(model)
T = 10  # fewer passes for demo speed
batch_size = 16

uncertainty_scores = []
features_list = []

with torch.no_grad():
    for i in range(0, n_pool, batch_size):
        batch = dummy_pool[i:i+batch_size]

        # Uncertainty: MC Dropout entropy
        prob_stack = mc_dropout_uncertainty(model, batch, T=T)
        entropy_map = compute_entropy(prob_stack)  # [B, H, W]
        img_entropy = entropy_map.mean(dim=[1, 2]).numpy()  # [B]
        uncertainty_scores.append(img_entropy)

        # Features: for diversity scoring
        feat_map = model.encode(batch)  # [B, 768, 16, 16]
        feat_vec = feat_map.mean(dim=[2, 3]).numpy()  # [B, 768]
        features_list.append(feat_vec)

uncertainty_scores = np.concatenate(uncertainty_scores)  # [100]
features = np.concatenate(features_list)  # [100, 768]

print(f"Uncertainty scores shape: {uncertainty_scores.shape}")
print(f"Features shape:           {features.shape}")

---
## 2. Combined Selection with Alpha

In [ ]:
# Select 20 samples with alpha=0.5 (equal weight)
budget = 20
alpha = 0.5

selected_indices, combined_scores = combined_selection(
    uncertainty_scores=uncertainty_scores,
    features=features,
    budget=budget,
    alpha=alpha,
    diversity_method="coreset",
    seed=42,
)

print(f"Selected {len(selected_indices)} samples with alpha={alpha}")
print(f"Indices: {selected_indices}")
print()
print(f"Combined scores for selected samples:")
for idx in selected_indices[:5]:
    print(f"  Image {idx}: combined={combined_scores[idx]:.4f}, "
          f"uncertainty={uncertainty_scores[idx]:.4f}")
print(f"  ... ({len(selected_indices) - 5} more)")

---
## 3. Effect of the Alpha Parameter

Let's see how different alpha values change the selection.

In [ ]:
# Sweep alpha from 0 (pure diversity) to 1 (pure uncertainty)
alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
selections = {}

for a in alphas:
    sel_idx, _ = combined_selection(
        uncertainty_scores=uncertainty_scores,
        features=features,
        budget=budget,
        alpha=a,
        diversity_method="coreset",
        seed=42,
    )
    selections[a] = set(sel_idx)

# Show overlap matrix
print("Overlap between selections at different alpha values:")
print(f"{'':>8}", end='')
for a in alphas:
    print(f"  a={a:.2f}", end='')
print()

for a1 in alphas:
    print(f"a={a1:.2f}", end='  ')
    for a2 in alphas:
        overlap = len(selections[a1] & selections[a2])
        print(f"  {overlap:5d}", end='')
    print()

In [ ]:
# Visualise: for each alpha, plot the uncertainty vs diversity of selected samples
from sklearn.manifold import TSNE
from scipy.spatial.distance import cdist

# Compute diversity scores as min-distance to already-selected set
# (simplified: we use the distance to the nearest neighbour in the pool)
pool_dists = cdist(features, features, metric='euclidean')
np.fill_diagonal(pool_dists, np.inf)
diversity_scores = pool_dists.min(axis=1)  # proxy for diversity

fig, axes = plt.subplots(1, len(alphas), figsize=(4 * len(alphas), 4), sharey=True)

for i, a in enumerate(alphas):
    ax = axes[i]
    sel = list(selections[a])
    unsel = [j for j in range(n_pool) if j not in sel]

    ax.scatter(uncertainty_scores[unsel], diversity_scores[unsel],
               c='lightgray', s=15, alpha=0.5)
    ax.scatter(uncertainty_scores[sel], diversity_scores[sel],
               c='#F44336', s=40, edgecolors='black', linewidth=0.5, zorder=5)
    ax.set_title(f'alpha={a:.2f}', fontsize=12)
    ax.set_xlabel('Uncertainty')
    if i == 0:
        ax.set_ylabel('Diversity (min-dist)')
    ax.grid(True, alpha=0.2)

fig.suptitle('Selected Samples at Different Alpha Values', fontsize=14, y=1.03)
fig.tight_layout()
plt.show()

print("alpha=0: selections cluster in the high-diversity region (spread out).")
print("alpha=1: selections cluster in the high-uncertainty region (hard samples).")
print("alpha=0.5: a balanced mix of both.")

---
## 4. Exporting the Selection as CSV

The selection is exported to a CSV file for the labeling session (US 5.4).
This CSV serves as the input to the human-in-the-loop step.

In [ ]:
# Export selection to CSV
output_path = Path("results/selection_round1.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

export_selection_csv(
    indices=selected_indices,
    image_paths=image_paths,
    scores=combined_scores,
    output_path=str(output_path),
)

print(f"Selection exported to: {output_path}")
print()

# Preview the CSV content
import csv
with open(output_path) as f:
    reader = csv.reader(f)
    for i, row in enumerate(reader):
        print(', '.join(row))
        if i >= 5:
            print('...')
            break

---
## 5. CLI Usage

```bash
# Combined selection (recommended)
udm-epic5 select \
    --strategy combined \
    --alpha 0.5 \
    --budget 50 \
    --checkpoint checkpoints/dann_best.pt \
    --config configs/epic5_active.yaml \
    --output results/selection_round1.csv

# Pure uncertainty
udm-epic5 select --strategy combined --alpha 1.0 --budget 50

# Pure diversity
udm-epic5 select --strategy combined --alpha 0.0 --budget 50
```

The output CSV columns: `index`, `image_path`, `combined_score`,
`uncertainty_score`, `diversity_score`.

---
## Summary

- The combined score = alpha * uncertainty + (1-alpha) * diversity.
- Alpha = 0.5 is a good default; tune based on your domain.
- Higher alpha focuses on hard samples; lower alpha focuses on coverage.
- The selection CSV feeds into the labeling session (US 5.4).

**Next:** `epic5_04_labeling.ipynb` -- set up a human-in-the-loop labeling
session for the selected samples.